# Flutter of the Heave-Pitch Typical Section with Quasi-Steady Aerodynamics

Welcome back! 👋

In our last notebook, we studied the heave-only typical section and found that the aerodynamics of an airfoil bouncing up and down acts purely as a **positive damper**: the system was always stable, no matter how fast we flew. Reassuring, but a bit boring. Stability at all speeds means no flutter!

So where does flutter come from? The missing ingredient is **coupling between heave and pitch**. When both degrees of freedom are active, the aeroelastic coupling forces a **phase relationship** between the two motions. At a critical airspeed, this phase alignment allows the aerodynamic forces to pump energy into the structure instead of removing it, and the oscillations grow without bound. This is the classical **binary flutter**: a dynamic aeroelastic instability arising from the interaction of two degrees of freedom.

In this notebook we combine heave $(h)$ and pitch $(\theta)$ motions to create a typical section model with 2 degrees of freedom (2-DOF) and witness this instability firsthand.

### Learning Objectives

By the end of this notebook, you will be able to:

1. Understand how heave and pitch motions couple mechanically through inertia.
2. Derive the equations of motion for the coupled system using Lagrange's equations and quasi-steady aerodynamics.
3. Understand the concept of vibration modes.
4. Formulate the state-space representation of the 2-DOF system, solve the eigenvalue problem and look for flutter.
5. Interpret root locus and V-g/V-f diagrams when flutter is present.
6. Explore how design parameters affect flutter speed.

_Let's get started!_ 🚀

In [ ]:
# We start by importing necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Automatically use "inline" for Google Colab or "widget" for interactive plots in Jupyter
import sys
if 'google.colab' in sys.modules:
    %matplotlib inline
else:
    %matplotlib widget

## 1. The Physical Setup: Heave and Pitch Degrees of Freedom and the Inertial Coupling

---

Look at our new configuration of the typical section below. This time, it's supported by two springs:
- $k_h$: The bending stiffness (resists vertical motion).
- $k_\theta$: The torsional stiffness (resists rotational motion).

Consequently, we have two degrees of freedom:
- Heave $h$: Vertical displacement (positive up).
- Pitch $\theta$: Angular rotation (positive nose-up).

<figure style="width:75%; margin: auto;">
<img src="../figures/04_typical_section_heave_pitch_dof.svg" style="width:100%">
<figcaption style="text-align: center;">

Typical section with heave and pitch degrees of freedom.

</figcaption>
</figure>

To study the behavior of this system, we define two reference frames:

- The **inertial frame** $\left(\hat{\boldsymbol{i}}_1, \hat{\boldsymbol{i}}_2, \hat{\boldsymbol{i}}_3\right)$ fixed to the ground.

- The **body frame** $\left(\hat{\boldsymbol{b}}_1, \hat{\boldsymbol{b}}_2, \hat{\boldsymbol{b}}_3\right)$ fixed to the airfoil and moving with it.

We set the origin of the body frame at the mid-chord point of the airfoil, and we take the half-chord length $b$ as our reference length (this is a typical choice in aeroelasticity).

On the section, we identify three notable points:
- **Aerodynamic Center (AC):** The point where the lift force can be considered to act, located at the quarter-chord of the airfoil.
- **Elastic Axis (EA):** The point about which the section twists, located at a distance $eb$ from the mid-chord point. In other words, $e$ represents the nondimensional offset of the EA from the mid-chord.
- **Center of Gravity (CG):** The point where we can consider the mass to be concentrated. We denote $x_\alpha b$ the distance of the CG from the elastic axis, where $x_\alpha$ is the nondimensional offset of the CG from the EA.

Finally, the section has the following inertial properties:
- $m$: Mass of the section.
- $I_\mathrm{CG}$: Mass moment of inertia about the CG.

### STOP and Think! 💭

If you move the CG up and down, does the section twist? And if you twist the section, does the CG move in the heave direction?

The idea is that, if the CG is located exactly at the EA, the two motions are independent. However, if the CG is offset by a distance from the EA, the two motions are coupled through inertial effects. This mechanical coupling is another type of coupling that we need to take into account in addition to the aeroelastic coupling that links the aerodynamic forces with the structural motion.

We are now going to derive the equations of motion for our system and see how these couplings manifest mathematically.

## 2. Derivation of the Equations of Motion

---

For a system with coupled degrees of freedom like ours, deriving equations of motion using force and moment equilibrium can become algebraically messy. We'd need to carefully track forces at multiple points and resolve them through the coupling geometry. Lagrange's equations offer a more systematic approach: we express the system's energy (kinetic and potential) and let the calculus machinery handle the derivatives. This energy-based method automatically accounts for all coupling terms and is particularly elegant for systems where constraints and multiple DOFs interact.

The approach is simple: we define your generalized coordinates $\boldsymbol{q}$ (in our case, $h$ and $\theta$), write down the kinetic energy $T$ and potential energy $U$, compute the Lagrangian $\mathscr{L} = T - U$, and then apply Lagrange's equations to get the equations of motion. Let's see how this works for our typical section.

We start from Lagrange's equations:

$$\frac{\mathrm{d}}{\mathrm{d}t}\left(\frac{\partial \mathscr{L}}{\partial \dot{\boldsymbol{q}}}\right) - \frac{\partial \mathscr{L}}{\partial \boldsymbol{q}} = \boldsymbol{Q},$$

where $\boldsymbol{Q}$ are the non-conservative generalized forces acting on our system. In our case, these are the aerodynamic forces that we will derive from the work done by the lift force through a virtual displacement of the AC.

The generalized coordinates for our system are:

$$\boldsymbol{q} = \begin{bmatrix} h \\ \theta \end{bmatrix}.$$

The potential energy of the system is given by the energy stored in the two springs:

$$U = \frac{1}{2}k_h h^2 + \frac{1}{2}k_\theta \theta^2.$$

The total kinetic energy of the system is given by the sum of the translational and rotational kinetic energies:

$$T = \frac{1}{2}m\boldsymbol{v}_\mathrm{CG} \cdot \boldsymbol{v}_\mathrm{CG} + \frac{1}{2}I_\mathrm{CG}\dot{\theta}^2,$$

where $\boldsymbol{v}_\mathrm{CG}$ is the velocity vector of the CG. The velocity vector of the CG is given by the vector sum of the heave velocity, which is evaluated at the EA, and of the tangential velocity due to pitch:

$$\boldsymbol{v}_\mathrm{CG} = \boldsymbol{v}_\mathrm{EA} + \boldsymbol{v}_t.$$

In terms of our generalized coordinates, the heave velocity at the EA is simply:

$$\boldsymbol{v}_\mathrm{EA} = \dot{h}\hat{\boldsymbol{i}}_3,$$

while the tangential velocity due to pitch is given by the cross product of the angular velocity vector and the position vector from the EA to the CG:

$$\boldsymbol{v}_t = \dot{\theta} \hat{\boldsymbol{b}}_2 \times x_\alpha b\, \hat{\boldsymbol{b}}_1 = - \dot{\theta} x_\alpha b\, \hat{\boldsymbol{b}}_3.$$

The minus sign in the above expression indicates that a positive pitch rate (nose-up) induces a downward tangential velocity at the CG for a positive $x_\alpha$ (CG behind the EA).

The unit vector $\hat{\boldsymbol{b}}_3$ has components in the inertial frame given by $\cos\theta\hat{\boldsymbol{i}}_3 + \sin\theta \hat{\boldsymbol{i}}_1$. Consequently, the velocity vector of the CG can be expressed in the inertial frame as:

$$\boldsymbol{v}_\mathrm{CG} = \dot{h}\hat{\boldsymbol{i}}_3 - \dot{\theta}x_\alpha b(\cos\theta\hat{\boldsymbol{i}}_3 + \sin\theta\hat{\boldsymbol{i}}_1).$$

To simplify the expression, we use the small angles assumption, which we use anyway to calculate our aerodynamic forces. If $\theta$ is small, then we can approximate $\cos\theta \approx 1$ and $\sin\theta \approx \theta$. Thus, the velocity vector of the center of mass simplifies to:

$$\boldsymbol{v}_\mathrm{CG} \approx \left(\dot{h} - \dot{\theta}x_\alpha b\right)\hat{\boldsymbol{i}}_3 - \dot{\theta}x_\alpha b\theta\hat{\boldsymbol{i}}_1.$$

The second term can be neglected in the assumption of small $\theta$, and consequently we can write the product $\boldsymbol{v}_\mathrm{CG} \cdot \boldsymbol{v}_\mathrm{CG}$ as:

$$\boldsymbol{v}_\mathrm{CG} \cdot \boldsymbol{v}_\mathrm{CG} \approx \left(\dot{h} - b\dot{\theta}x_\alpha\right)^2 = \dot{h}^2 - 2bx_\alpha\dot{h}\dot{\theta} + b^2x_\alpha^2\dot{\theta}^2.$$

Substituting this expression into the kinetic energy equation, we obtain:

$$T = \frac{1}{2}m\left(\dot{h}^2 - 2bx_\alpha\dot{h}\dot{\theta} + b^2x_\alpha^2\dot{\theta}^2\right) + \frac{1}{2}I_\mathrm{CG}\dot{\theta}^2 = \frac{1}{2}m\left(\dot{h}^2 - 2bx_\alpha\dot{h}\dot{\theta}\right) + \frac{1}{2}I_\mathrm{EA}\dot{\theta}^2,$$

where $I_\mathrm{EA} = I_\mathrm{CG} + m\left(bx_\alpha\right)^2$ is the mass moment of inertia about the EA, obtained using the parallel axis theorem.

The Lagrangian $\mathscr{L} = T - U$ can now be expressed as:

$$\mathscr{L} = \frac{1}{2}m\left(\dot{h}^2 - 2bx_\alpha\dot{h}\dot{\theta}\right) + \frac{1}{2}I_\mathrm{EA}\dot{\theta}^2 - \frac{1}{2}k_h h^2 - \frac{1}{2}k_\theta \theta^2.$$

We then take the partial derivatives of the Lagrangian with respect to the generalized coordinates and their time derivatives:

$$\frac{\partial \mathscr{L}}{\partial h} = -k_h h, \quad \frac{\partial \mathscr{L}}{\partial \theta} = -k_\theta \theta,$$

$$\frac{\partial \mathscr{L}}{\partial \dot{h}} = m\dot{h} - mbx_\alpha \dot{\theta}, \quad \frac{\partial \mathscr{L}}{\partial \dot{\theta}} = I_\mathrm{EA}\dot{\theta} - mbx_\alpha \dot{h}.$$

Successively, we compute the time derivatives $\frac{\mathrm{d}}{\mathrm{d}t}\left(\frac{\partial \mathscr{L}}{\partial \dot{\boldsymbol{q}}}\right)$:

$$\frac{\mathrm{d}}{\mathrm{d}t}\left(\frac{\partial \mathscr{L}}{\partial \dot{h}}\right) = m\ddot{h} - mbx_\alpha \ddot{\theta},$$

$$\frac{\mathrm{d}}{\mathrm{d}t}\left(\frac{\partial \mathscr{L}}{\partial \dot{\theta}}\right) = I_\mathrm{EA} \ddot{\theta} - mbx_\alpha \ddot{h}.$$

The generalized forces associated with the degrees of freedom $h$ and $\theta$ can be derived from the work done by the aerodynamic lift through a virtual displacement and a virtual rotation about its point of application, that is to say the AC. Mathematically, the generalized forces are defined as:

$$\boldsymbol{Q} = \frac{\partial (\delta W)}{\partial (\delta \boldsymbol{q})}.$$

As we have seen in Notebook 2, we can assume quasi-steady aerodynamics to compute the lift force based on the istantaneous value of the angle of attack. Furthermore, by using the heave approximation, we have also seen that the heave motion induces an effective angle of attack given by $\alpha_h \approx -\dot{h}/V$. For our 2-DOF model, we need to account for the contribution of both the static angle $\theta$ and the effective angle of attack due to heave $\alpha_h$ to compute the total angle of attack $\alpha$:

$$\alpha = \theta + \alpha_h = \theta - \frac{\dot{h}}{V}.$$

Note that, with the pitch motion, there is also an additional contribution to the effective angle of attack due to the tangential velocity induced by the pitch rate $\dot{\theta}$. However, for the moment we assume that this contribution is negligible, and we will come back to it in the next notebook.

The lift force can thus be expressed as:

$$
L = 2 q b c_{l\alpha} \left(\theta - \frac{\dot{h}}{V}\right),
$$

where $q = \frac{1}{2}\rho V^2$ is the dynamic pressure, $2b$ is the chord length, and $c_{l\alpha} = 2\pi$ is the lift curve slope for a symmetric thin airfoil.

To find the work done by the lift, we need to compute the virtual displacement of the AC. Similarly to what we did for the CG, the velocity vector of the AC is given by the sum of the heave velocity at the EA and of the tangential velocity due to pitch:

$$
\boldsymbol{v}_\mathrm{AC} = \dot{h}\hat{\boldsymbol{i}}_3 + \dot{\theta}\hat{\boldsymbol{b}}_2 \times \left(-\frac{b}{2} - eb\right)\hat{\boldsymbol{b}}_1 = \dot{h}\hat{\boldsymbol{i}}_3 + b\left(\frac{1}{2} + e\right)\dot{\theta}\hat{\boldsymbol{b}}_3,
$$

where this time the plus sign of the tangential velocity term indicates an upward motion of the AC due to positive pitch rate (nose-up) when the AC is ahead of the EA $\left(e>-1/2\right)$.

Analogously to the velocity vector of the CG, in the small angle assumption this expression simplifies to:

$$\boldsymbol{v}_\mathrm{AC} \approx \left(\dot{h} + b\dot{\theta}\left(\frac{1}{2} + e\right)\right)\hat{\boldsymbol{i}}_3.$$

The virtual displacement of the AC can be obtained by replacing the time derivatives of the generalized coordinates with their virtual displacements:

$$
\delta\boldsymbol{p}_\mathrm{AC} = \left(\delta h + b\delta\theta\left(\frac{1}{2} + e\right)\right)\hat{\boldsymbol{i}}_3.
$$

The work done by the aerodynamic lift through the virtual displacement of the AC is then given by:

$$
\delta W = L\hat{\boldsymbol{i}}_3 \cdot \delta\boldsymbol{p}_\mathrm{AC} = 2 q b c_{l\alpha} \left(\theta - \frac{\dot{h}}{V}\right)\left(\delta h + b\delta\theta\left(\frac{1}{2} + e\right)\right).
$$

Taking the partial derivatives with respect to $\delta h$ and $\delta \theta$, we find the generalized forces:

$$Q_h = \frac{\partial (\delta W)}{\partial (\delta h)} = 2 q b c_{l\alpha} \left(\theta - \frac{\dot{h}}{V}\right) = L$$

$$Q_\theta = \frac{\partial (\delta W)}{\partial (\delta \theta)} = 2 q b c_{l\alpha} \left(\theta - \frac{\dot{h}}{V}\right) b\left(\frac{1}{2} + e\right) = L b\left(\frac{1}{2} + e\right).$$

In other words, the generalized forces associated with the heave and pitch degrees of freedom are respectively given by the lift force and by the moment generated by the lift force about the EA. In this case the result is pretty intuitive and we could have guessed it without going through Lagrange's equations and the principle of virtual work, but the beauty of this approach is that it works even when the forces are not as straightforward to interpret.

Substituting the expressions for the Lagrangian derivatives and the generalized forces into Lagrange's equations, we obtain the equations of motion for the typical section with heave and pitch degrees of freedom:

$$m\ddot{h} - mbx_\alpha \ddot{\theta} + k_h h = 2 q b c_{l\alpha} \left(\theta - \frac{\dot{h}}{V}\right),$$

$$I_\mathrm{EA} \ddot{\theta} - mbx_\alpha \ddot{h} + k_\theta \theta = 2 q b^2 c_{l\alpha} \left(\theta - \frac{\dot{h}}{V}\right) \left(\frac{1}{2} + e\right).$$

Let's stop for a moment and think about these equations. We can immediately make two observations:
- The equations are coupled through the inertial terms involving the product $mbx_\alpha$. As we predicted intuitively, this means that even in the absence of aerodynamic forces, the heave and pitch motions influence each other due to the offset between the CG and the EA.
- The equations are also coupled through the aerodynamic terms on the right-hand side. This is the aeroelastic coupling that we have already seen our previous notebooks, where the aerodynamic forces depend on the structural motion.

We also note that this is a system of two coupled second-order linear ordinary differential equations with constant coefficients and that we can rewrite it in homogeneous form by bringing all terms to the left-hand side. Substituting $q = \frac{1}{2}\rho V^2$ and $c_{l\alpha} = 2\pi$ to simplify the expressions we obtain:

$$m\ddot{h} - mbx_\alpha \ddot{\theta} + 2\pi \rho b V \dot{h} + k_h h - 2\pi \rho b V^2 \theta = 0,$$

$$I_\mathrm{EA} \ddot{\theta} - mbx_\alpha \ddot{h} + 2\pi \rho b^2 V \left(\frac{1}{2} + e\right) \dot{h} + \left(k_\theta - 2\pi \rho b^2 V^2 \left(\frac{1}{2} + e\right)\right) \theta = 0.$$

Once again, let's stop for a moment and think about the structure of these equations. We can see that they have the general form of a coupled mass-spring-damper system, where:
- The terms involving $\ddot{h}$ and $\ddot{\theta}$ represent the inertial forces.
- The terms involving $\dot{h}$ represent the aerodynamic damping forces. Remember that we neglected the contribution of the pitch rate $\dot{\theta}$ to the effective angle of attack, so this is the reason why there is no damping term involving $\dot{\theta}$ in the equations.
- The terms involving $h$ and $\theta$ represent the elastic forces, given by the structural stiffness and the aerodynamic stiffness.

We can express the system in matrix form as:

$$
\boldsymbol{M}\ddot{\boldsymbol{q}} + \boldsymbol{C}\dot{\boldsymbol{q}} + \boldsymbol{K}\boldsymbol{q} = \boldsymbol{0},
$$

where the mass matrix $\boldsymbol{M}$, the damping matrix $\boldsymbol{C}$, and the stiffness matrix $\boldsymbol{K}$ are given by:

$$
\boldsymbol{M} = \begin{bmatrix}
m & -mbx_\alpha \\
-mbx_\alpha & I_\mathrm{EA}
\end{bmatrix}, \quad
\boldsymbol{C} = \begin{bmatrix}
2\pi \rho b V & 0 \\
2\pi \rho b^2 V \left(\frac{1}{2} + e\right) & 0
\end{bmatrix}, \quad
\boldsymbol{K} = \begin{bmatrix}
k_h & -2\pi \rho b V^2 \\
0 & k_\theta -2\pi \rho b^2 V^2 \left(\frac{1}{2} + e\right)
\end{bmatrix}.
$$

**Important observations:**

1. The mass matrix $\boldsymbol{M}$ is symmetric and contains off-diagonal terms that represent the inertial coupling between heave and pitch due to the offset of the CG from the EA.

2. Notice how the damping matrix $\boldsymbol{C}$ has **zeros in the second column**. This means there's no direct damping term proportional to $\dot{\theta}$ in either equation, as a consequence of having neglected the pitch rate effects to the effective angle of attack. The only source of aerodynamic damping is the heave velocity $\dot{h}$.

3. The damping matrix $\boldsymbol{C}$ and stiffness matrix $\boldsymbol{K}$ contain one **off-diagonal term** each that couples the heave and pitch motions aeroelastically. The off-diagonal term in the damping matrix tells us that the heave velocity $\dot{h}$ induces a damping moment in the pitch equation, while the off-diagonal term in the stiffness matrix tells us that the pitch angle $\theta$ induces a stiffness force in the heave equation.

4. The lower-right term of $\boldsymbol{K}$ contains the **summation of a structural and an aerodynamic contribution**, totally analogous to what we had in Notebook 1 when we derived the equilibrium equation of the static pitch-only typical section. This means that what we studied in Notebook 1 is actually embedded in this more general model, meaning that also the 2-DOF system can experience divergence.

5. Both the damping and stiffness matrices **depend on flight speed** $V$. This is the hallmark of aeroelastic systems: the dynamic characteristics change with operating conditions!

This matrix formulation is compact and elegant, but similarly to Notebook 2, in order to solve it computationally and understand its stability, we need to transform it into state-space form. Let's do that next.

## 3. State-Space Representation: Setting Up for Eigenvalue Analysis

---

We have a second-order system of differential equations. In order to study the stability of our system computationally, we need to convert it into a first-order system, just as we did for the heave-only model in Notebook 2.

### Converting to State-Space Form

We define a **state vector** $\boldsymbol{x}$ that includes both the displacements and velocities:

$$
\boldsymbol{x} = \begin{bmatrix} h \\ \theta \\ \dot{h} \\ \dot{\theta} \end{bmatrix}
$$

This means that our state vector has **4 components** for a 2-DOF system (compared to 2 components in the 1-DOF case of Notebook 2). Starting from our matrix equation:

$$\boldsymbol{M}\ddot{\boldsymbol{q}} + \boldsymbol{C}\dot{\boldsymbol{q}} + \boldsymbol{K}\boldsymbol{q} = \boldsymbol{0},$$

we solve for the accelerations:

$$\ddot{\boldsymbol{q}} = -\boldsymbol{M}^{-1}\boldsymbol{C}\dot{\boldsymbol{q}} - \boldsymbol{M}^{-1}\boldsymbol{K}\boldsymbol{q},$$

and write:

$$
\dot{\boldsymbol{x}} = \boldsymbol{A}\boldsymbol{x},
$$

where the **system matrix** $\boldsymbol{A}$ is:

$$
\boldsymbol{A} = \begin{bmatrix}
\boldsymbol{0}_{2\times 2} & \boldsymbol{I}_{2\times 2} \\
-\boldsymbol{M}^{-1}\boldsymbol{K} & -\boldsymbol{M}^{-1}\boldsymbol{C}
\end{bmatrix}
$$

This is a $4 \times 4$ matrix. The identity matrix $\boldsymbol{I}$ in the upper right block ensures that the first two components of $\dot{\boldsymbol{x}}$ correspond to the velocities $\dot{h}$ and $\dot{\theta}$.

### Eigenvalues, Eigenvectors, and the Search for Flutter

The stability of our system is determined by the **eigenvalues** of $\boldsymbol{A}$. In Notebook 2 we had a $2\times 2$ system matrix with 2 eigenvalues. Now our $4\times 4$ matrix gives us **4 eigenvalues**, which for an underdamped system come in **2 complex conjugate pairs**.

For each eigenvalue pair $\lambda = \lambda_r \pm i\lambda_i$:
- The **real part** $\lambda_r$ determines stability:
  - $\lambda_r < 0$: the mode is **stable** (decays)
  - $\lambda_r = 0$: **neutrally stable** (constant amplitude)
  - $\lambda_r > 0$: **unstable** (grows exponentially) ⚠️
- The **imaginary part** $\lambda_i$ gives the **oscillation frequency**

Each eigenvalue also has an associated **eigenvector** $\hat{\boldsymbol{x}}$, which tells us the *shape* of the corresponding vibration mode, that is to say the relative amplitude and phase of the heave and pitch components. We'll come back to eigenvectors shortly.

Because the matrices $\boldsymbol{C}$ and $\boldsymbol{K}$ depend on flight speed $V$, **the system matrix $\boldsymbol{A}$ changes with $V$**, and so do its eigenvalues. Our strategy is therefore to compute the eigenvalues at many flight speeds and track how they move in the complex plane. **Flutter** is the event we are looking for: the speed at which an eigenvalue's real part crosses from negative to positive, signalling the onset of instability.

### Organizing model parameters with Python `dataclass`

To solve the eigenvalue problem numerically, we need concrete values for all the physical parameters of our typical section. In Notebook 2 we listed every parameter as a keyword argument of the function `build_system_matrix`. That works well when there are few parameters, but the 2-DOF model already needs 8 parameters, and as we increase the complexity of the typical section models that we consider, we will need even more parameters. Long argument lists become hard to read and easy to get wrong.

Python's `dataclass` (available since Python 3.7, from the `dataclasses` module in the standard library) solves this by letting us group related parameters into a single object. Think of it as a lightweight container: we declare the fields (with names, types, and default values), and Python automatically generates the constructor (the function to instantiate the object) and a readable string representation for us.

**How it works:**

```python
from dataclasses import dataclass

@dataclass
class TypicalSectionParams:
    c: float = 1.6               # chord length [m]
    e: float = -0.1              # EA position from midchord [in semichords]
    x_alpha: float = 0.1         # CG position from EA [in semichords]
    # ... more fields ...
```

The `@dataclass` line above the class definition is called a **decorator** and it tells Python to automatically add useful features to the class (like a constructor that accepts each field as a keyword argument, and a `__repr__` method that prints the object nicely).

**How to use it:**

```python
# Create an instance with all default values
params = TypicalSectionParams()

# Access a field
print(params.e)   # → -0.1

# Create an instance where only the elastic axis position differs
params_custom = TypicalSectionParams(e=0.3)

# Access the field with custom value
print(params_custom.e)   # → 0.3
```

**Why we use it here:** In this notebook you'll have to analyze the sensitivity of the 2-DOF model to a number of design parameters. Instead of calling the function `build_system_matrix` with all the specific input arguments each time, we create a separate parameter object. Each parameter object is self-contained, and `build_system_matrix` receives everything it needs through a single argument. This makes our code cleaner and more maintainable.

As you might have noticed in the code snippet above, each field of the `dataclass` can have a default value, which we use to define the baseline configuration of our 2-DOF model:

- Chord $c=1.6$ m
- Nondimensional EA position $e=-0.1$ (EA slightly ahead of the mid-chord)
- Nondimensional CG position $x_\alpha=0.2$ (CG behind the EA)
- Mass $m=200$ kg/m
- Mass moment of inertia about the EA $I_\mathrm{EA}=30$ kg·m²/m
- Heave stiffness $k_h=4 \cdot 10^4$ N/m
- Pitch stiffness $k_\theta=9 \cdot 10^4$ N·m/rad
- Lift curve slope $c_{l\alpha}=2\pi$ 1/rad

Note that air density $\rho$ is **not** included in the dataclass. Since it is a property of the flow (set by the altitude), not of the typical section itself, we keep it as a separate input argument of `build_system_matrix`, with default value equal to the sea level density $\rho = 1.225$ kg/m³.

In [ ]:
from dataclasses import dataclass  # import the dataclass decorator from the standard library


@dataclass  # this decorator automatically generates __init__ and __repr__ methods
class TypicalSectionParams:
    """Physical parameters for the 2-DOF heave-pitch typical section model."""
    # Geometrical properties
    c: float = 1.6               # chord length [m]
    e: float = -0.1              # EA position from midchord [in semichords]
    x_alpha: float = 0.2         # CG position from EA [in semichords]
    
    # Inertial properties
    m: float = 200.0             # mass per unit span [kg/m]
    I_ea: float = 30.0           # mass moment of inertia about EA [kg*m^2/m]

    # Elastic properties
    k_h: float = 4e4             # heave stiffness [N/m]
    k_theta: float = 9e4         # torsional stiffness [N*m/rad]

    # Aerodynamic properties
    cl_alpha: float = 2 * np.pi  # lift curve slope [1/rad]

Did we choose these numbers at random? Not really. We've tried to replicate what would be representative first bending and torsion frequencies of a commercial aircraft wing. As a quick sanity check, let's compute the natural frequencies of the **mechanically uncoupled** system **in vacuum**, i.e. an imaginary version of our model at zero speed (no aerodynamic forces) and where the CG sits exactly on the EA $(x_\alpha = 0)$, so that heave and pitch are independent. In this case, the natural frequencies are simply:

$$
\omega_h = \sqrt{\frac{k_h}{m}}, \quad \omega_\theta = \sqrt{\frac{k_\theta}{I_\mathrm{EA}}}.
$$

In [ ]:
# Create baseline parameter set
params = TypicalSectionParams()

# Calculate natural frequencies of the mechanically uncoupled system
omega_h_uncoupled = np.sqrt(params.k_h / params.m)  # heave natural frequency [rad/s]
omega_theta_uncoupled = np.sqrt(params.k_theta / params.I_ea)  # pitch natural frequency [rad/s]

# Print the natural frequencies in both rad/s and Hz (recall: f = omega / 2*pi)
print("--- Mechanically Uncoupled Frequencies (as if x_alpha = 0) ---") 
print(f"Heave natural frequency: {omega_h_uncoupled:.1f} rad/s, {omega_h_uncoupled / (2 * np.pi):.2f} Hz")
print(f"Pitch natural frequency: {omega_theta_uncoupled:.1f} rad/s, {omega_theta_uncoupled / (2 * np.pi):.2f} Hz")
print(f"Frequency ratio omega_theta / omega_h = {omega_theta_uncoupled / omega_h_uncoupled:.2f}")

Indeed, typical first bending and torsion frequencies of a commercial aircraft wing fall respectively in the range of 1–3 Hz and 6–10 Hz, although of course these values can vary significantly depending on the specific aircraft. However, these frequencies are NOT the actual characteristic frequencies of the coupled system, and we've just used them for a quick and dirty assessment. What if we want to know the actual frequencies of the coupled system? Let's find out!

### Vibration Modes of the Coupled System

Our actual system has $x_\alpha = 0.2 \neq 0$, which means heave and pitch are **mechanically coupled** through inertia even before we switch on the aerodynamics. This has an important consequence: the natural motions of the coupled system are no longer "pure heave" and "pure pitch" and we can no longer calculate the natural frequencies with the simple formulas above.

To understand what happens, recall what you may have seen in a basic vibration course for a 2-DOF mass-spring-damper system. When you displace one mass and let go, both masses start moving. There exist special initial conditions for which both masses oscillate at the same frequency and maintain a fixed amplitude ratio. These special motions are the **vibration modes** (or **normal modes**) of the system. For example, in a 2-DOF mass-spring-damper system, you typically have two modes:
- A **lower-frequency mode** where the two masses move in phase.
- A **higher-frequency mode** where the two masses move out of phase.

Each mode is characterised by:
- A **characteristic frequency**: how fast the mode oscillates.
- A **damping ratio**: how fast the mode decays (or grows, if unstable).
- A **mode shape**: the relative amplitude and phase of each degree of freedom in that mode.

Knowing the vibration modes of a system is crucial for understanding its dynamic behavior and it allows us to describe the system's response to any initial condition as a combination of these modes.

For our typical section, finding the modes at zero flight speed corresponds to finding the modes of the undamped system, as aerodynamics is the only source of damping, and we can do it by solving the following **generalized eigenvalue problem**:

$$\boldsymbol{K}_0 \hat{\boldsymbol{x}}_j = \omega_j^2 \boldsymbol{M} \hat{\boldsymbol{x}}_j$$

where $\boldsymbol{K}_0 = \text{diag}(k_h, k_\theta)$ is the structural stiffness matrix, and $\omega_j$ is the frequency of the $j$-th mode. Since $\boldsymbol{M}$ is not diagonal (because $x_\alpha \neq 0$), the mode shapes will not correspond to pure heave or pure pitch. Instead, each mode will contain **both** heave and pitch, although one component typically dominates, which is why we refer to them as the **heave-dominated** and **pitch-dominated** modes.

Let's compute them and compare their frequencies with those of the uncoupled system.

In [ ]:
# We define a function to calculate the natural frequencies and mode shapes of the coupled system at V=0
# This function builds the mass and stiffness matrices of the system at zero speed, solves the generalised
# eigenvalue problem, and returns the natural frequencies and mode shapes.
def calculate_coupled_frequencies_and_modes(params):
    """
    Calculate the natural frequencies and mode shapes of the coupled heave-pitch system at V=0.
    
    Parameters:
        params (TypicalSectionParams): The parameters of the typical section model.
    
    Returns:
        omega (ndarray): The natural frequencies of the coupled system [rad/s].
        eigenvectors (ndarray): The mode shapes of the coupled system, with each column corresponding to a mode.
    """
    # Build mass and structural stiffness matrices at V=0
    M = np.array([
        [params.m,                                 -params.m * params.c/2 * params.x_alpha],
        [-params.m * params.c/2 * params.x_alpha,  params.I_ea                            ]
    ])
    K0 = np.diag([params.k_h, params.k_theta])  # numpy diag function creates a diagonal matrix from the given list of diagonal elements

    # Solve the generalised eigenvalue problem: K0 x_hat = omega^2 M x_hat
    # This is equivalent to finding the eigenvalues of M^{-1} K0
    M_inv = np.linalg.inv(M)
    eigenvalues, eigenvectors = np.linalg.eig(M_inv @ K0)

    # Sort by ascending frequency (ascending order of eigenvalues)
    sort_idx = np.argsort(eigenvalues)
    eigenvalues = eigenvalues[sort_idx]
    eigenvectors = eigenvectors[:, sort_idx]  # the eig function returns eigenvectors as columns, so we sort the columns according to the sorted indices

    # The eigenvalues correspond to omega^2, so we take the square root to get the frequencies
    omega = np.sqrt(eigenvalues)

    # Return the frequencies and mode shapes
    return omega, eigenvectors


# We feed the baseline parameters into the function to get the frequencies and mode shapes of the coupled system at V=0
omega_coupled, mode_shapes = calculate_coupled_frequencies_and_modes(params)

# Print the frequencies and mode shapes of the coupled system
print("--- Mechanically Coupled Frequencies (at V = 0) ---")

# Iterate over the two modes
for j in range(2):

    # Extract the mode shape for the j-th mode
    x_hat = mode_shapes[:, j]

    # Determine if the mode is heave-dominated or pitch-dominated based on the relative magnitudes of the components
    mode_label = "Heave-dominated" if np.abs(x_hat[0]) >= np.abs(x_hat[1]) else "Pitch-dominated"

    # Print the frequency and mode shape for the j-th mode
    print(f"\nMode {j+1} ({mode_label}):")
    print(f"  Frequency: {omega_coupled[j]:.1f} rad/s ({omega_coupled[j]/(2*np.pi):.2f} Hz)")
    print(f"  Mode shape [h, theta]: [{x_hat[0]:.4f}, {x_hat[1]:.4f}]")

# Compare the frequencies of the coupled system with those of the uncoupled system
print(f"\n--- Comparison ---")
print(f"Uncoupled frequencies: {omega_h_uncoupled/(2*np.pi):.2f} Hz and {omega_theta_uncoupled/(2*np.pi):.2f} Hz")
print(f"Coupled frequencies:   {omega_coupled[0]/(2*np.pi):.2f} Hz and {omega_coupled[1]/(2*np.pi):.2f} Hz")

A few observations:
- Both modes contain a mixture of heave and pitch. As we can observe from the components of the mode shapes, the lower-frequency mode is dominated by heave, while the higher-frequency mode is dominated by pitch, but in both cases the other component is not zero. This means that if we displace the system according to one of these modes, we will see a combination of heave and pitch motion.
- The **coupled frequencies** differ slightly from the uncoupled ones, and the difference on the higher-frequency pitch-dominated mode is more pronounced.
- Any free vibration of our system at $V = 0$ is a superposition of these two modes.

When we switch on the aerodynamics, we need to compute the eigenvalues and eigenvectors of the full system matrix $\boldsymbol{A}$ to find the aeroelastic modes of the system. As we will see, both the frequencies and the shapes of the initial modes at $V=0$ will evolve with flight speed. As aerodynamics introduces damping, the modes will also acquire a damping ratio, and the mode shapes will become complex, meaning that the relative phase between the heave and pitch components will no longer be 0 or 180 degrees, but something in between. Flutter will occur when at least one of the modes becomes unstable, that is to say when the real part of its eigenvalue becomes positive.

Finally, recall that, since $\boldsymbol{A}$ is a $4\times 4$ matrix, each eigenvector is a $4$-dimensional vector, including both the generalized coordinates and their time derivatives:

$$\hat{\boldsymbol{x}} = \begin{bmatrix} \hat{h} \\ \hat{\theta} \\ \hat{\dot{h}} \\ \hat{\dot{\theta}} \end{bmatrix}.$$

When we talk about the mode shape, we refer to the first two components of the eigenvector, which give us the relative amplitude and phase of the heave and pitch displacements in that mode.

### Building the System Matrix

We're now ready to implement the function that assembles the system matrix $\boldsymbol{A}$ for a given set of system parameters and a given flight condition. The function takes as input a `TypicalSectionParams` object, the flight speed $V$, and the air density $\rho$, and it returns the system matrix $\boldsymbol{A}$. 

In [ ]:
def build_system_matrix(params, velocity, rho=1.225):
    """
    Build the system matrix for the 2-DOF heave-pitch eigenvalue analysis.

    Parameters
    ----------
    params : TypicalSectionParams
        Physical parameters of the typical section.
    velocity : float
        Freestream velocity [m/s]
    rho : float
        Air density [kg/m^3]

    Returns
    -------
    system_matrix : ndarray, shape (4, 4)
        State-space system matrix A
    """
    # Derived quantities
    b = params.c / 2

    # Mass matrix (symmetric)
    M = np.array([
        [params.m,                        -params.m * b * params.x_alpha],
        [-params.m * b * params.x_alpha,  params.I_ea                   ]
    ])

    # Damping matrix
    # Note: the second column is zero because we are neglecting pitch rate effects
    C = np.array([
        [params.cl_alpha * rho * b * velocity,                        0],
        [params.cl_alpha * rho * b**2 * velocity * (0.5 + params.e),  0]
    ])

    # Stiffness matrix (note the negative off-diagonal term from aerodynamic coupling)
    K = np.array([
        [params.k_h,  -params.cl_alpha * rho * b * velocity**2                                        ],
        [0,            params.k_theta - params.cl_alpha * rho * b**2 * velocity**2 * (0.5 + params.e)]
    ])

    # Invert the mass matrix
    M_inv = np.linalg.inv(M)

    # Build the state matrix in block form
    zeros = np.zeros((2, 2))
    identity = np.eye(2)

    # Top row blocks: [0, I]
    top = np.hstack([zeros, identity])

    # Bottom row blocks: [-M^(-1)K, -M^(-1)C]
    bottom = np.hstack([-M_inv @ K, -M_inv @ C])

    # Stack vertically
    system_matrix = np.vstack([top, bottom])

    # Return the system matrix
    return system_matrix

## 4. Root Locus Analysis: Finding the Flutter Speed

---

Similarly to our last notebook, we compute the eigenvalues over a range of flight speeds and see how they evolve in the complex plane. Remember: the left half of the complex plane (negative real part) corresponds to stable modes, while the right half (positive real part) corresponds to unstable modes. Let's sweep a range of flight speeds between 0 and 150 m/s and see what happens!

In [ ]:
# Range of velocities to analyze
velocities = np.linspace(1, 150, 200)

# Initialize arrays to store real and imaginary parts of eigenvalues
# To plot the root locus, we will use the pyplot function scatter, which expects 1D arrays
# We know there will be 4 eigenvalues for each velocity, so we preallocate a 1D array with shape (N_velocities * 4,)
N_EIGENVALUES = 4
real_parts = np.zeros((len(velocities) * N_EIGENVALUES,))
imag_parts = np.zeros((len(velocities) * N_EIGENVALUES,))

# Iterate over velocities
for idx, V in enumerate(velocities):

    # Build system matrix
    A = build_system_matrix(params, V)

    # Compute eigenvalues
    eigenvalues = np.linalg.eigvals(A)

    # Store real and imaginary parts
    # Since we have 4 eigenvalues for each velocity, we store them in the appropriate slice of the preallocated arrays
    real_parts[idx * N_EIGENVALUES : (idx + 1) * N_EIGENVALUES] = eigenvalues.real
    imag_parts[idx * N_EIGENVALUES : (idx + 1) * N_EIGENVALUES] = eigenvalues.imag

# Create figure
plt.figure()

# Scatter plot: x=Real, y=Imaginary, Color=Speed
# The arrays including the real and imaginary parts have shape (N_velocities * 4,)
# In order to color the points according to velocity, we need to repeat each velocity four times to match the structure of the real_parts and imag_parts arrays
# We use the repeat function from NumPy to achieve this
# The function call np.repeat(velocities, 4) creates an array where each element of the velocity array is repeated four consecutively
# The scatter function returns a PathCollection object that we can use to create the colorbar
sc = plt.scatter(real_parts, imag_parts, c=np.repeat(velocities, N_EIGENVALUES))

# Set plot elements
plt.xlabel("$\\mathbb{Re}(\\lambda)$ - Exponential Decay/Growth Rate")
plt.ylabel("$\\mathbb{Im}(\\lambda)$ - Oscillation Frequency, rad/s")
plt.grid(True)

# Add colorbar and show plot
plt.colorbar(sc, label="$V$, m/s")
plt.show()

**Stop and Think.** Let's observe a few things from this root locus plot:
- We now have 4 eigenvalue trajectories, corresponding to 4 eigenvalues of our 2-DOF system.
- The eigenvalues appear in two complex conjugate pairs, as expected for an underdamped 2-DOF system.
- Each pair corresponds to a vibration mode of the system. The mode with lower frequency corresponds to the heave-dominated motion, while the mode with higher frequency corresponds to the pitch-dominated motion.
- Both modes start close to the imaginary axis at low speeds, as aerodynamic forces are the only source of damping in our system, and with no aerodynamics the system is undamped.
- The real part of the heave-dominated mode becomes more negative with increasing speed, indicating that it becomes more stable, while its frequency remains relatively constant.
- The pitch-dominated mode, on the other hand, shows a different behavior as speed increases. Initially the real part becomes more negative, indicating increased stability, but at a certain speed, it starts to move towards the right half of the complex plane, until it crosses into the unstable region. The speed at which this happens is the **flutter speed** of the system, and the corresponding imaginary part gives us the **flutter frequency**. Furthermore, the frequency of the pitch-dominated mode decreases with increasing speed, showing a more pronounced effect of aerodynamic stiffness compared to the heave-dominated mode.

### V-g and V-f Diagrams

Similarly to our last notebook, we are now going to use the two companion plots of the root locus diagram:

1. **V-g diagram**: Damping ratio vs. velocity
   - Shows how stable/unstable each mode is
   - **Flutter speed** = where damping crosses zero

2. **V-f diagram**: Frequency vs. velocity
   - Shows how mode frequencies change with speed
   - **Flutter frequency** = frequency at flutter speed

Compared to the root locus, these plots make it easier to read off the critical flutter speed and frequency, and to understand how the dynamic characteristics of each mode evolve with flight speed.

In [ ]:
# Initialize arrays to store frequency and damping ratio
# Since we have two degree of freedom, there will be two frequencies and two damping ratios per velocity
frequencies = np.zeros((len(velocities), 2))  # these will be angular frequencies omega [rad/s], with shape (N_velocities, 2)
damping_ratios = np.zeros((len(velocities), 2))

# We reshape the arrays of the real and imaginary parts from shape (N_velocities * 4,) to shape (N_velocities, 4)
# so that we can easily access the eigenvalues corresponding to each velocity
real_parts = real_parts.reshape((len(velocities), N_EIGENVALUES))
imag_parts = imag_parts.reshape((len(velocities), N_EIGENVALUES))

# We iterate over the velocities to calculate frequency and damping ratio
for idx in range(len(velocities)):

    # Sort by highest imaginary part (frequency)
    # Complex conjugate pairs will have equal and opposite imaginary parts
    # We sort in ascending order to have the negative frequencies first and the positive frequencies last
    sort_idx = np.argsort(imag_parts[idx])

    # Keep only the eigenvalues with the two highest frequencies
    # The two highest frequencies will be the last two elements in the sorted order, which correspond to the positive frequencies
    sorted_real = real_parts[idx][sort_idx[-2:]]
    sorted_imag = imag_parts[idx][sort_idx[-2:]]

    # Store current imaginary parts in row idx of frequencies array
    # Given how we sorted the eigenvalues, we know that the first column of the frequencies array will
    # correspond to the heave-dominated mode and the second column will correspond to the pitch-dominated mode
    frequencies[idx] = sorted_imag
    
    # Calculate Damping Ratio zeta
    damping_ratios[idx] = -sorted_real / np.sqrt(sorted_real**2 + sorted_imag**2)

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

# Plot 1: V-g plot (damping vs velocity)
# Since velocity is a 1D array of shape (N_velocities,) and damping_ratios is a 2D array of shape (N_velocities, 2),
# the plot function will automatically plot the two columns of the damping_ratios array as two separate lines
ax1.plot(velocities, damping_ratios)
ax1.set_ylabel('$\\zeta$')
ax1.grid(True)

# Plot 2: V-f plot (frequency vs velocity)
ax2.plot(velocities, frequencies)
ax2.set_ylabel('$\\omega$, rad/s')
ax2.set_xlabel('$V$, m/s')
ax2.grid(True)

# Add legend to the first plot
# In this case we know that the first column of the damping_ratios array corresponds to the heave-dominated mode
# and the second column corresponds to the pitch-dominated mode, because we sorted the eigenvalues by their imaginary
# part (frequency) and we know from our V=0 analysis that the heave-dominated mode has lower frequency than the
# pitch-dominated mode. If for any reason we use typical section parameters that flip the order of the modes at V=0,
# we would need to adjust the legend accordingly.
ax1.legend(['Heave-dominated mode', 'Pitch-dominated mode'])

# Set tight layout and show plot
plt.tight_layout()
plt.show()

**Interpreting the V-g Diagram:**

- The heave-dominated mode becomes more stable with increasing speed (damping ratio increases).
- The pitch-dominated mode initially becomes more stable, but then its damping ratio decreases and eventually becomes negative, indicating the onset of flutter. The **flutter speed** $V_F$ is where the damping curve of the pitch-dominated mode crosses zero.

**Interpreting the V-f Diagram:**

- The frequency of the heave-dominated mode remains relatively constant with increasing speed, showing only a slight increase.
- The frequency of the pitch-dominated mode decreases significantly with increasing speed.
- The two modes **approach each other** in frequency as speed increases, which is a common precursor to flutter. When modes get close in frequency, they can interact in such a way that leads to instability.
- The **flutter frequency** $\omega_F$ is the frequency corresponding to the flutter speed $V_F$ identified in the V-g diagram.

Before we delve deeper into how the two modes interact by looking at how the mode shapes evolve with speed, there's an exercise four you! Can you identify the flutter speed and flutter frequency by looking at the `damping_ratios` and `frequencies` arrays that we computed in the code above? Fill in the code cell below and print your results!

In [ ]:
# Find index of first negative damping ratio for the pitch-dominated mode (which is the second column of the damping_ratios array)
# You can use the np.where function to find the indices where the condition is true, and then take the first one with [0][0]:
# np.where(...)[0][0] gives you the first index where the condition is true
# ...

# Find flutter speed as the average of the two velocities around the flutter index (the one just before and just after the damping ratio becomes negative)
# ...

# Find flutter frequency as the average of the two frequencies around the flutter index
# ...

# Print results
# ...

### How Mode Shapes Evolve with Flight Speed

Now let's look at how the mode shapes evolve with flight speed. Remember that each eigenvalue has an associated eigenvector, which gives us the relative amplitude and phase of the heave and pitch components in that mode.

At $V = 0$ the system is undamped and the eigenvectors are **real**: the $h$ and $\theta$ components are either perfectly in phase (same sign) or perfectly out of phase (opposite signs). But as soon as the aerodynamics kicks in ($V > 0$), damping enters the picture and the eigenvectors become **complex**. A complex eigenvector means that the $h$ and $\theta$ components of a mode are no longer exactly in phase or out of phase, and there is a **phase lag** between them. Physically, this means that the heave and pitch motions reach their peak amplitudes at slightly different instants within each oscillation cycle.

We can represent each component of a complex eigenvector as a **magnitude** and a **phase angle**: the magnitude tells us how large that component is relative to the other, and the phase angle tells us how much it leads or lags.

Let's compute and print the mode shapes at three representative speeds: near zero, at half the flutter speed, and at the flutter speed.

In [ ]:
# Select three representative velocities for mode shape analysis
eigvec_velocities = np.array([1, flutter_speed / 2, flutter_speed])

# Preallocate array: for each velocity, we store 2 modes × 2 DOF components (complex)
mode_shapes = np.zeros((len(eigvec_velocities), 2, 2), dtype=complex)  # shape (N_velocities, N_dofs, N_modes)

# Iterate over the selected velocities
for i, V in enumerate(eigvec_velocities):
    # Build system matrix at this speed
    A = build_system_matrix(params, V)

    # Compute eigenvalues and eigenvectors
    eigenvalues, eigenvectors = np.linalg.eig(A)

    # Sort by imaginary part (ascending) so that positive-frequency modes come last
    sort_idx = np.argsort(eigenvalues.imag)

    # Keep only the two modes with positive frequency (last two after sorting)
    eigenvalues = eigenvalues[sort_idx[-2:]]
    eigenvectors = eigenvectors[:, sort_idx[-2:]]

    # Extract the displacement part of each eigenvector (first 2 components: h and theta)
    mode_shapes[i] = eigenvectors[:2, :]

    # Normalize each mode shape to have maximum magnitude of 1 for easier comparison
    # The norm function with axis=0 computes the norm of each column (mode shape) separately, resulting in an
    # array of norms for each mode shape
    mode_shapes[i] = mode_shapes[i] / np.max(np.abs(mode_shapes[i]), axis=0)
    # The line above is equivalent to:
    # mode_shapes[i, :, 0] = mode_shapes[i, :, 0] / np.linalg.norm(mode_shapes[i, :, 0])
    # mode_shapes[i, :, 1] = mode_shapes[i, :, 1] / np.linalg.norm(mode_shapes[i, :, 1])

    # Print the results for the current velocity
    print(f"\n{'='*50}")
    print(f"V = {V:.1f} m/s")
    print(f"{'='*50}")

    # Iterate over the two modes
    for j in range(2):
        # Extract the mode shape for the j-th mode
        x_hat = mode_shapes[i, :, j]

        # Determine if the mode is heave-dominated or pitch-dominated
        mode_label = "Heave-dominated" if np.abs(x_hat[0]) >= np.abs(x_hat[1]) else "Pitch-dominated"

        print(f"\n  Mode {j+1} ({mode_label}):")
        print(f"    Frequency: {eigenvalues[j].imag:.1f} rad/s ({eigenvalues[j].imag/(2*np.pi):.2f} Hz)")

        # Print each DOF component as magnitude and phase
        dof_names = ['h    ', 'theta']
        for k in range(2):
            magnitude = np.abs(x_hat[k])
            phase_deg = np.degrees(np.angle(x_hat[k]))
            print(f"    {dof_names[k]}:  magnitude = {magnitude:.4f},  phase = {phase_deg:+.1f}°")

Let's unpack what these numbers are telling us.

**At $V \approx 0$ m/s** (nearly undamped system): all phase angles are close to $\pm 90°$. This means each DOF is either in phase or exactly out of phase with the dominant component. The eigenvectors are essentially real, just as we found from the generalised eigenvalue problem at $V=0$. The mode shapes look like the vibration modes we computed earlier.

**At $V = V_F/2$** (moderate aerodynamic damping for both modes): notice that some phase angles have drifted away from $\pm 90°$. In the heave-dominated mode, for instance, the $h$ component has shifted to a more negative angle, and the $\theta$ component has shifted to a more positive angle. This **phase lag** is the fingerprint of aerodynamic damping: it means the pitch displacement reaches its peak at a slightly different moment than the heave displacement within each oscillation cycle. Furthermore, we can observe that the magnitude of the $\theta$ component in the heave-dominated mode has increased compared to $V \approx 0$, while the magnitude of the $h$ component in the pitch-dominated mode has decreased. This is a sign that the two modes are indeed changing with speed.

**At $V = V_F$** (flutter onset, pitch-dominated mode undamped): the phase lag has grown further. Also the magnitude of the $\theta$ component in the heave-dominated mode has increased further, and the magnitude of the $h$ component in the pitch-dominated mode has almost disappeared, meaning that the mode is almost pure pitch at the flutter speed.

To better visualize this evolution, let's represent each mode shape as a pair of arrows (phasors) in the complex plane, one for $h$ and one for $\theta$. The **length** of each arrow is the component's magnitude, and the **angle** from the positive real axis is its phase. This is exactly the same information as above, but plotted rather than printed.

In [ ]:
# Get default matplotlib color cycle (we'll use the first two colors for the two modes)
COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Create a figure with three subplots (one per velocity), arranged vertically
fig, axes = plt.subplots(3, 1, figsize=(5, 15))

# Define angle array to draw a unit circle as a visual reference
angle_array_circle = np.linspace(0, 2 * np.pi, 100)

# Iterate over the selected velocities
for idx, V in enumerate(eigvec_velocities):

    # Draw unit circle for reference (dashed gray line)
    axes[idx].plot(np.cos(angle_array_circle), np.sin(angle_array_circle), "k--", alpha=0.3, linewidth=1)

    # Iterate over the two modes
    for j in range(2):

        # Extract the mode shape for the j-th mode
        x_hat = mode_shapes[idx, :, j]

        # Determine if the mode is heave-dominated or pitch-dominated
        mode_label = "Heave-dominated" if np.abs(x_hat[0]) >= np.abs(x_hat[1]) else "Pitch-dominated"

        # Iterate over the two components of the mode shape (h and theta)
        dof_labels = ["$h$", "$\\theta$"]
        for k in range(2):

            # Extract the real and imaginary parts of the k-th component
            real_part = np.real(x_hat[k])
            imag_part = np.imag(x_hat[k])

            # Plot the mode shape component as an arrow (phasor) in the complex plane
            axes[idx].arrow(
                0,  # Start x-coordinate
                0,  # Start y-coordinate
                real_part,  # End x-coordinate (real part of the mode shape component)
                imag_part,  # End y-coordinate (imaginary part of the mode shape component)
                head_width=0.05,   # Size of the arrowhead
                head_length=0.05,  # Size of the arrowhead
                fc=COLORS[j],      # Fill color matching the mode
                ec=COLORS[j],      # Edge color matching the mode
                linewidth=2,       # Line width of the arrow
                zorder=2,          # Ensure arrows are plotted above the unit circle
                label=f"{mode_label} mode" if k == 0 else None  # label only the first DOF to avoid duplicate legend entries
            )

            # Add text label to indicate the degree of freedom (h or theta)
            text_x = real_part + .1 * np.sign(real_part) if k == 0 else real_part - .1 * np.sign(real_part)
            text_y = imag_part + .07 * np.sign(imag_part) if imag_part != 0 else -.07
            axes[idx].text(
                text_x,
                text_y,
                dof_labels[k],
                color=COLORS[j],
                fontsize=9,
                ha="center",
                va="center",
            )

    # Set subplot formatting
    axes[idx].set_title(f'$V={V:.1f}$ m/s')
    axes[idx].set_xlabel('Real Part')
    axes[idx].set_ylabel('Imaginary Part')
    axes[idx].set_aspect('equal')  # ensure circles look circular
    axes[idx].set_xlim(-1.1, 1.1)
    axes[idx].set_ylim(-1.1, 1.1)
    axes[idx].grid(True)
    axes[idx].legend()

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

Each subplot shows the two modes as pairs of phasors (arrows). The blue arrows belong to the heave-dominated mode, the orange arrows to the pitch-dominated mode. Let's read them:

**At $V \approx 0$ m/s:** all arrows point almost straight up or down (i.e. nearly purely real). Within each mode, the two DOFs are either in phase ($0°$ apart) or out of phase ($180°$ apart). This is what we expect for the undamped structural modes.

**At $V = V_F/2$:** the arrows within each mode are no longer perfectly aligned or opposite. Focus on the heave-dominated mode (blue): the $\theta$ arrow has rotated away from the $\pm 180°$ direction with respect to the $h$ arrow. This angular difference represents the **phase lag** between heave and pitch within this mode, caused by the aerodynamic damping.

**At $V = V_F$:** the phase lag within the heave-dominated mode has grown further, together with the magnitude of the $\theta$ arrow. At the same time we can see that the $h$ arrow of the pitch dominated mode has almost disappeared.

In summary, the phasor plots give us a visual interpretation on what we call "mode interaction". As speed increases, both the frequency and the mode shapes become more similar, as both modes see a significant increase in their relative pitch component. At a certain speed, the pitch-dominated mode becomes unstable, and this is what triggers flutter.

## 5. Time-Domain Simulations: Seeing Flutter in Action

---

Eigenvalue analysis tells us *when* flutter occurs, but to really **see** what flutter looks like, we need to simulate the time response of our system. In order to do this, we need to set up the function that computes the time derivative of the state vector, which we can then integrate over time using `odeint`, like we did in our last notebook.

In [ ]:
# Define the function that returns dx/dt = A * x
def state_space_model(x, time_array, system_matrix):
    """
    Computes the derivative of the state vector.

    Parameters:
    x (array): State vector [h, theta, h_dot, theta_dot]
    time_array (float): Time (not used here because our system is linear time-invariant, but odeint requires it)
    system_matrix (matrix): System matrix

    Returns:
    dxdt (array): Time derivative of the state vector
    """
    dxdt = system_matrix @ x
    return dxdt

Let's now pick three representative flight speeds:

1. **Well below flutter speed**: 60 m/s, we expect a stable, well-damped motion
2. **Just below flutter speed**: 105 m/s, we expect lightly damped, slow decaying motion
3. **Above flutter speed**: 120 m/s, we expect unstable, exponentially growing oscillations ⚠️

We'll give the system a small initial disturbance and watch what happens. Careful: this time we have two degrees of freedom, so we need to set initial conditions for both heave and pitch and we will have to plot both responses!

In [ ]:
# Define flight speeds to test
flight_speeds = [60, 105, 120]

# Define simulation parameters
t = np.linspace(0, 3, 1000)  # Simulate for 3 seconds
initial_condition = [0.5, 0.0, 0.0, 0.0]  # Initial displacement h = 0.5 m, theta = 0 rad, h_dot = 0 m/s, theta_dot = 0 rad/s

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

# Iterate over speeds
for V in flight_speeds:

    # Build the system matrix for this speed
    A = build_system_matrix(params, V)

    # Solve the ODE
    solution_array = odeint(state_space_model, initial_condition, t, args=(A,))

    # Extract 'h' (the first column of the solution)
    h_response = solution_array[:, 0]
    theta_response = solution_array[:, 1]

    # Plot
    ax1.plot(t, h_response, label=f"$V = {V}$ m/s")
    ax2.plot(t, np.degrees(theta_response), label=f"$V = {V}$ m/s")

# Set axes labels, legend, and grid
ax2.set_xlabel("$t$, s")
ax1.set_ylabel("$h$, m")
ax2.set_ylabel("$\\theta$, deg")
ax1.legend()
ax1.grid(True)
ax2.grid(True)
plt.show()

What do you see? As usual, let's make some observations.
- At 60 m/s, both heave and pitch motions show damped oscillations that decay to zero over time, indicating a stable system. As predicted by the V-g plot, the damping of the pitch-dominated mode is lower than that of the heave-dominated mode, resulting in a slower decay of the pitch motion.
- At 105 m/s, the heave motion decays even faster than at 60 m/s. Indeed the damping of the heave-dominated mode at this speed is higher. The pitch motion, on the other hand, shows almost no decay, indicating that the system is close to neutral stability. This is consistent with the V-g plot, where the damping of the pitch-dominated mode is very close to zero at this speed.
- At 120 m/s, the pitch motion grows exponentially over time, indicating an unstable system. This is consistent with the fact that the pitch-dominated mode has become unstable at this speed, as shown in the V-g plot. The heave motion instead appears to decay initially, as the heave-dominated mode is still stable. However, after this initial transient, also the heave motion starts to grow, as it is driven by the heave-component of the unstable pitch-dominated mode.

## 6. The Physical Mechanism of Flutter 🌀

---

Before we move to exercises, let's take a moment to understand **why** flutter happens. It's not just mathematics, there's an interesting physical mechanism at work.

For the case of classical bending-torsion flutter this mechanism is well illustrated in the figure below. The figure describes the interaction between the bending and torsional motions of a cantilevered wing, respectively corresponding to the heave and pitch motions of our typical section, assuming that the lift is dependent only on the static angle of attack. This is a further simplification compared to our model where we also included the contribution of the heave velocity. However, the physical mechanism is the same, and the figure is very useful to understand how the coupling between heave and pitch can lead to instability.

1. **Frequencies approach**: The bending and torsional oscillation frequencies get close to each other as speed increases.
   - This is exactly what we see in our V-f plot for the heave- and the pitch-dominated modes!
   - The figure shows the limit case where the frequencies of bending and torsion are exactly the same.

2. **Phase Relationship**: The bending and torsional motions develop a specific phase relationship.
   - When the wing pitches nose-up, it's also moving upward, and when it pitches nose-down, it's moving downward.
   - In this way, the lift force generated by the pitch motion is in phase with the heave motion.

3. **Energy Transfer**: The coupled motion extracts energy from the airflow.
   - The aerodynamic forces do **positive work** on the system.
   - This compensates for (and eventually exceeds) structural damping.

4. **Self-Excitation**: The system becomes a self-excited oscillator.
   - The amplitude grows until limited by nonlinear effects or structural failure.

<figure style="width:75%; margin: auto;">
<img src="../figures/04_demasi_flutter.png" style="width:100%">
<figcaption style="text-align: center;">

Illustration of the physical mechanism behind classical bending-torsion flutter. Reproduced from "Introduction to Unsteady Aerodynamics and Dynamic Aeroelasticity", Luciano Demasi, Springer (2024).

</figcaption>
</figure>

In reality, the frequencies of the bending and torsional modes don't need to be exactly the same for flutter to occur, as we've seen in our typical section model, but the physical mechanism remains the same. When the modes get close enough in frequency, they can interact in such a way to develop a phase relationship that allows the aerodynamic forces to do a net positive work on the system, potentially leading to instability if the structural damping is not sufficient to counteract this transfer of energy.

## 7. Exercises: Exploring Design Parameter Effects on Flutter

---

Now it's your turn to explore! Flutter speed is sensitive to many design parameters. Understanding these sensitivities is crucial for aircraft design.

Since `build_system_matrix` takes a `TypicalSectionParams` object, you can easily create modified parameter sets to investigate different configurations:

```python
# Example: change EA position
params_modified = TypicalSectionParams(e=0.1)

# Example: change CG position relative to EA
params_modified = TypicalSectionParams(x_alpha=0.3)
```

### Exercise 1: Effect of EA Position

In Notebook 1 we saw how the position of the elastic axis influences the divergence speed. In particular, we found that as we move the EA further behind the AC the divergence speed decreases, while as we move it closer to the AC the divergence speed increases, until we reach a position of the EA ahead of the AC where divergence disappears. Does the same happen for flutter? How does the flutter speed change as we move the EA further behind or closer to the AC?

1. Start from the baseline parameters and try increasing the value of $e$ to 0.1 and 0.2 (EA further behind the AC). What happens to the flutter speed?
2. Now try decreasing $e$ compared to the baseline, e.g. -0.2 and -0.3 (EA closer to the AC). What happens?
3. Finally, decrease the value of $e$ to -0.6, which corresponds to the EA being ahead of the AC. Does flutter still occur?

### Exercise 2: Effect of CG Position

The position of the centre of gravity relative to the elastic axis is a critical parameter for flutter. Before proceeding with the exercise, try to predict intuitively how the flutter speed will change as we move the CG further behind or closer to the EA. Then, verify your predictions by running the following configurations:

1. Start with the baseline parameters and try increasing the value of $x_\alpha$ to 0.3 and 0.4 (CG further behind the EA). What happens to the flutter speed?
2. Now try decreasing $x_\alpha$ compared to the baseline, e.g. 0.1 and 0.05 (CG closer to the EA). What happens?
3. Finally, set $x_\alpha = 0$, which corresponds to the CG sitting exactly on the EA, and try also a negative value (CG ahead of the EA). What happens to our system? And can you think about the physical mechanism behind what you observe?

### Exercise 3: Effect of Initial Frequency Ratio

The ratio between the zero-speed frequencies of the two modes, $\omega_\theta / \omega_h$, is a key factor in determining the flutter speed. When these frequencies are close to each other, the modes can start interacting at lower speeds, which typically leads to a lower flutter speed. Conversely, when the frequencies are far apart, the modes tend to interact later as speed increases, and flutter tends to occur at higher speeds. You can change the initial frequency ratio by modifying the stiffness and the inertia parameters of the system. Your task is to explore how increasing and decreasing each of these parameters affects the initial frequency ratio and the flutter speed. Repeat the following steps for each parameter among $k_h$, $m$, $k_\theta$, and $I_\mathrm{EA}$:

1. Increase the parameter and observe how the initial frequency ratio changes. What happens to the flutter speed?
2. Decrease the parameter and observe how the initial frequency ratio changes. What happens to the flutter speed?

What different trends do you observe for stiffness and inertia parameters? Do you notice anything that seems counterintuitive at first?

Remember to use the initial frequencies of the coupled system to calculate the initial frequency ratio, not the uncoupled frequencies!

### Suggested Approach

To ease your exploration, I suggest that you build a single code cell that outputs all the information and plots of interest for a given set of parameters, and then you can simply modify the parameters and re-run the cell to see how the results change. Here's a suggested structure for your code cell:

- Start with the definition of the `TypicalSectionParams` instance, which you can modify to explore different configurations.
- Calculate the initial frequencies (V=0 m/s) of the coupled system using the function `calculate_coupled_frequencies_and_modes` and print the initial frequency ratio.
- Copy the code for sweeping velocities, building the system matrix, computing eigenvalues, and plotting the root locus from Section 4.
- Adapt the code that we used to find the flutter speed from the V-g diagram. Calculate this value from the results of the root locus analysis and print it out.

Once you have this code cell set up, you can easily modify the parameters and re-run it to see how the flutter speed changes with different design choices. This will give you a deeper understanding of the sensitivity of flutter to various parameters and the underlying physical mechanisms at play.

In [ ]:
## ============================================================
## EXERCISE SOLUTION TEMPLATE
## Change the line below to explore different configurations
## ============================================================
params_ex = TypicalSectionParams()  # <-- modify parameters here!

# --- Step 1: Compute initial coupled frequencies ---
# Calculate the coupled frequencies using the function calculate_coupled_frequencies_and_modes
# Print initial frequencies and their ratio (omega_heave/omega_pitch)
# ...

# --- Step 2: Sweep velocities and compute eigenvalues ---
# Use the same velocity range as before or define a new one if you want to explore a different range
# Initialize the arrays to store the real and imaginary parts of the eigenvalues for the new parameter set
# ...

# Iterate over the velocities
# ...
    # Build the system matrix for the current velocity and parameter set
    # ...
    # Compute the eigenvalues of the system matrix
    # ...
    # Store the real and imaginary parts in the appropriate slices of the preallocated arrays
    # ...

# --- Step 3: Plot root locus ---
# Create a new figure and axis for the root locus plot
# ...

# Call the scatter function to plot the real vs imaginary parts of the eigenvalues, coloring by velocity
# ...

# Set labels, title, grid, and colorbar, and show the plot
# ...

# --- Step 4: Compute flutter speed ---
# Reshape the real part array to have shape (N_velocities, N_EIGENVALUES) for easier analysis
# ...

# Find the index of the first row of the real part array where any of the eigenvalues has a positive real part (indicating instability)
# You can use np.where to find the indices where the condition is true, and then take the first one with [0][0]
# ...

# Print the flutter speed corresponding to this index, which is the average of the two velocities around the flutter index
# ...

## 8. Summary and a Look Ahead 🎓

---

Congratulations! You've just completed your introduction to flutter. Let's recap what you've learned:

### Key Takeaways

1. **Coupling is Essential**: Flutter requires coupling between heave and pitch through an aeroelastic mechanism (aerodynamic forces depending on both motions).

2. **Vibration Modes**: The coupled heave-pitch system has two vibration modes, each containing both heave and pitch participation. Inertial coupling ($x_\alpha \neq 0$) mixes the modes, and aerodynamic forces modify them further as speed increases. At nonzero speed, the eigenvectors become complex, encoding a **phase lag** between heave and pitch within each mode.

3. **State-Space Formulation & Eigenvalue Analysis**: Converting the 2nd-order ODE to 1st-order state-space form enables eigenvalue analysis. A 2-DOF system gives 4 eigenvalues in 2 complex conjugate pairs (assuming the system is underdamped). The root locus and the V-g and V-f diagrams are complementary ways to track how these eigenvalues evolve with speed and to identify the flutter boundary.

5. **Physical Mechanism**: Flutter arises from frequency approach between two modes combined with a phase relationship that allows the coupled motion to extract energy from the airflow.

6. **Design Parameters Matter**: Flutter speed is sensitive to EA and CG position, and to the initial frequency ratio (closer frequencies → easier flutter).

### What We Haven't Covered:

- **Unsteady Aerodynamics**: More accurate modeling of unsteady aerodynamic forces
  - Theodorsen's function for oscillating airfoils
  - Wagner's function for arbitrary motion
  - Finite-state approximations (e.g., Roger's approximation)

- **Multiple Degrees of Freedom**: Real aircraft wings
  - Finite element models
  - Modal analysis and reduction

- **Nonlinear Effects**: What happens after flutter onset
  - Limit cycle oscillations
  - Structural nonlinearities

But you now have the fundamental tools to understand these more advanced topics in the future!

### Final Thought 💭

Aeroelastic instabilities were one of the major challenges in the early days of aviation. The understanding you've gained today represents decades of research that enabled safe high-speed flight. Every time you fly, remember that countless hours of aeroelastic analyses have ensured your safety.

**Well done!** 🎉